# Event-Conditioned Persistent Neural PDE–ODE Dynamics for Irregularly Observed Music Forecasting

*An executable research article and falsifiable Google Colab benchmark · V3 · 14 November 2026*

## Abstract

Learning temporally coherent musical representations from irregularly observed audio remains difficult because time–frequency spectra contain smooth harmonic evolution, short-lived transients, and structure at disparate temporal scales. Initial one-step experiments with coupled neural partial differential equations and ordinary differential equations (PDE–ODEs) failed to demonstrate a meaningful improvement over persistence and diffusion-free ablations. We investigate a different formulation, termed **event-conditioned persistent coupled dynamics** (ECPD), in which a mask-aware history encoder initializes a global latent state, an event-rate predictor estimates future transient forcing from *past observations alone*, and a spectral field and the global state are jointly integrated across an entire forecasting horizon. The field evolves under learned convolutional reactions, optional nonnegative diffusion, global latent feedback, and predicted event-conditioned forcing. We formulate a recording-disjoint benchmark with randomly missing observations, progressively longer horizons, and evaluations of spectral error, onset-proxy fidelity, and high-frequency degradation. Competing architectures include a neural ODE, temporal CNN, GRU, Transformer, persistence, linear extrapolation, and three pretrained Hugging Face forecasting checkpoints: TinyTimeMixer R2, Chronos-Bolt-Tiny and Chronos-T5-Tiny. The hypotheses are that persistent latent evolution and event forcing improve dynamic prediction under partial observations; results are not presumed. This notebook implements the experimental protocol and saves measured outcomes, including failed pretrained-model attempts. It does **not** claim music understanding, a physical diffusion law for spectra, or state-of-the-art forecasting performance.

## 1. Research gap, questions, and scope

**Observed negative result.** Earlier eight-frame rollouts produced near-identical errors for the coupled model, ablations and persistence. Crucially, the earlier latent variable was reset after every step and training used a one-frame objective. These defects prevent the experiment from testing persistent dynamics.

- **RQ1 — persistence:** does jointly integrating field and global state over 8–96 frames reduce error relative to the same architecture without persistent global dynamics?
- **RQ2 — event structure:** does *history-predicted* transient forcing improve onset-proxy F1 and high-frequency fidelity without sacrificing smooth-region accuracy?
- **RQ3 — missing observations:** can a time-gap- and mask-aware model generalize to observation rates different from the training distribution?
- **RQ4 — cross-recording generalization:** does the proposed method outperform independently trained GRU, temporal CNN, Transformer and neural ODE on entire unseen recordings?
- **RQ5 — pretrained transfer:** how does it compare with three pretrained time-series forecasters when those models are transferred *out of domain* to audio-frequency frames?

**Expected outcomes (hypotheses, not findings):** event forcing should matter primarily near transients; persistence should matter increasingly with horizon; explicit missingness features should help at low observation rates. Any hypothesis may be rejected by the data.

**Claim boundary:** frequency-axis diffusion is a modeling assumption, and spectral-flux detections are proxy events, not human note annotations. The synthetic dataset is for reproducibility and debugging; real-music claims require licensed recording-disjoint evaluation.

## 2. Proposed method

Let \(u(t)\in\mathbb R^F\) be the standardized log-spectral field, \(z(t)\in\mathbb R^d\) a persistent global state, and \(\hat e(t)\) a *predicted* event forcing obtained solely from irregular history. We integrate both state variables across the full future time grid:

\[
\dot u = D_\theta\Delta_f u + R_\theta(u) + C_\theta z + \hat e(t)q_\theta,\qquad
\dot z = G_\theta\big(z,\operatorname{pool}(u),\hat e(t)\big),
\quad D_\theta\ge 0.
\]

The encoder observes forward-filled historical frames \(\tilde x_j\), binary observation indicators \(m_j\), and elapsed time since last observation \(\delta_j\), and produces \(z(0)\) and a future event-rate schedule. The initial field is the last available observed spectrum. **One ODE solve returns the complete forecast; no future true frames or future event labels enter inference.**

### 2.1. What is different from the failed approach?

The earlier system fitted next-frame regression, reset latent memory in each forecasting step, and used a very short horizon. ECPD retains latent state, optimizes the complete long-horizon trajectory, is conditioned on observed time gaps, and predicts event forcing without test-time privileged information. The three ablations **without event forcing**, **without diffusion**, and **without dynamic global feedback** attribute gains to individual mechanisms. A separate neural ODE tests whether a PDE structure is necessary at all.

### 2.2. Expected outcomes and falsification

We do not expect the proposed model to universally win. Event forcing may lead to false positives; diffusion may blur partials; integration may be more expensive than direct sequence forecasting. The primary test is mean squared error over the entire forecast, with onset-proxy F1 and high-frequency MSE as co-primary diagnostic outcomes. **Claim improvement only if the method beats strong baselines on held-out recordings with adequate uncertainty and acceptable compute cost.**

## 3. Experimental workflow and proposed-method diagrams

The executable figures distinguish the **proposed mechanism** from baseline predictors. Thick blue nodes designate the research intervention; neutral nodes are shared data and validation stages. Highlighting does not encode an assumed win. The evaluation chart later uses a separate outline for ECPD and a correctly scaled horizon-wise plot so that a divergent baseline does not visually obscure small differences.

In [ ]:
%pip -q install torchdiffeq scipy pandas matplotlib graphviz "chronos-forecasting>=2.1,<3" "transformers>=4.45,<5"

In [ ]:
import os, math, json, time, random, copy, traceback, gc, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import stft, resample_poly
from scipy.io import wavfile
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchdiffeq import odeint

try:
    from graphviz import Digraph
    from IPython.display import display
    g=Digraph('proposed_method',format='png')
    g.attr(rankdir='LR',fontsize='12')
    for k,t in [('record','Recording-disjoint data'),('irregular','Missing observations\nmask + time gap'),('encode','Mask-aware history encoder'),('events','History-predicted\nevent schedule'),('ode','PROPOSED joint PDE-ODE\n(u,z) full horizon'),('verify','Validation and independent test')]:
        g.node(k,t,shape='box',style='rounded,filled' if k in ('events','ode') else 'rounded',fillcolor='#dbeafe' if k in ('events','ode') else 'white',color='#1d4ed8' if k in ('events','ode') else '#64748b',penwidth='3' if k in ('events','ode') else '1')
    for a,b in [('record','irregular'),('irregular','encode'),('encode','events'),('events','ode'),('ode','verify')]:g.edge(a,b)
    display(g)
except Exception as exc:
    print('Diagram rendering unavailable:',exc)

## 4. Dataset, irregular-observation mechanism, and recording-disjoint splits

Recordings are transformed into 24-band log-magnitude STFT features at a 128-sample hop with an 8-kHz sampling rate. All windows of one file belong to only one partition, and normalization is estimated from the training recordings only. The optional ZIP must contain legally usable WAV recordings organized as `genre/*.wav` or `instrument/*.wav`. If absent, a deterministic multi-instrument, multi-tempo synthetic set is generated and marked **synthetic** in all outputs.

**Irregularly observed inputs** are created by dropping historical frames at nonuniform intervals. The model sees their observation mask and age (seconds since last valid observation); missing frames are forward-filled without accessing future targets. All competing models receive the same observed, imputed history; pretrained forecasters cannot use the mask and age channels and are analyzed separately.

Use `FAST_DEV=True` for a Colab smoke test. Increase the seeds, epochs and number of real recordings for a paper; execution time can increase considerably.

In [ ]:
SEED=20261114
FAST_DEV=True
SEEDS=[11] if FAST_DEV else [11,23,37,53,71,89,101,127,139,151]
EPOCHS=2 if FAST_DEV else 18
BATCH=8 if FAST_DEV else 16
SR=8000; HOP=128; FFT=512; BANDS=24; POOL=8
CONTEXT=512; HMAX=96; HORIZONS=[8,24,48,96]
MODEL_STRIDE=4  # all trainable models see the same 128 temporal positions from 512 historical frames
DT=HOP/SR
WINDOW_STRIDE=48 if FAST_DEV else 24
TRAIN_KEEP_RANGE=(0.35,0.85)
TEST_KEEP_RATES=[1.0,0.65,0.35]
USE_REAL_WAV_ZIP=False
MAX_RECORDINGS=24 if FAST_DEV else 60
RECORDING_SECONDS=21 if FAST_DEV else 30
OUT=Path('music_ecpd_v3_results');OUT.mkdir(exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device',device,'data mode:',('WAV ZIP' if USE_REAL_WAV_ZIP else 'SYNTHETIC'), 'seeds:',SEEDS)

In [ ]:
def synth_track(seed, instrument, tempo, seconds=RECORDING_SECONDS):
    rng=np.random.default_rng(seed)
    x=np.zeros(int(seconds*SR),dtype=np.float32)
    harmonics={'soft':[1,.15,.04,.01], 'bright':[1,.75,.4,.22],
               'reed':[1,.75,.13,.48], 'plucked':[1,.5,.22,.08]}[instrument]
    beat=60.0/tempo
    base=float(rng.choice([110,146.83,164.81,196,220,261.63]))
    for i,start in enumerate(np.arange(0,seconds,beat)):
        pos=int(start*SR);length=min(int(beat*SR),len(x)-pos)
        if length<=0:continue
        t=np.arange(length)/SR
        pitch=base*2**(int(rng.choice([-5,-3,0,2,4,7]))/12)
        envelope=np.minimum(t*25,1)*np.exp(- (3.0 if instrument=='plucked' else 0.7)*t)
        part=sum(a*np.sin(2*np.pi*(k+1)*pitch*t) for k,a in enumerate(harmonics))
        x[pos:pos+length]+=envelope*part
        # Predictable but nontrivial transients, with occasional syncopation.
        if i%3==0:
            kick_len=min(240,len(x)-pos)
            x[pos:pos+kick_len]+=(.08*rng.standard_normal(kick_len)*np.exp(-np.arange(kick_len)/35)).astype(np.float32)
    x+=.002*rng.standard_normal(len(x)).astype(np.float32)
    x/=max(1e-6,float(np.max(np.abs(x))))
    return x

def load_wav(path):
    sr,x=wavfile.read(path)
    x=np.asarray(x,dtype=np.float32)
    if x.ndim>1:x=x.mean(axis=1)
    x/=max(float(np.max(np.abs(x))),1e-6)
    if sr!=SR:
        k=math.gcd(int(sr),SR)
        x=resample_poly(x,SR//k,int(sr)//k)
    return x[:int(RECORDING_SECONDS*SR)]

def spectrogram(x):
    _,_,z=stft(x,fs=SR,nperseg=FFT,noverlap=FFT-HOP,boundary=None)
    # Skip DC; 24 groups × 8 bins, so output is fixed-dimensional.
    spectrum=np.abs(z[1:1+BANDS*POOL]).reshape(BANDS,POOL,-1).mean(axis=1)
    return np.log1p(25*spectrum).T.astype(np.float32)

records=[]
if USE_REAL_WAV_ZIP:
    from google.colab import files
    import zipfile
    archive=files.upload();file_name=next(iter(archive))
    import io
    with zipfile.ZipFile(io.BytesIO(archive[file_name])) as zf:
        for item in zf.infolist():
            if len(records)>=MAX_RECORDINGS:break
            from pathlib import PurePosixPath
            path=PurePosixPath(item.filename)
            if item.is_dir() or path.suffix.lower()!='.wav' or '..' in path.parts:continue
            import tempfile
            with tempfile.NamedTemporaryFile(suffix='.wav') as fp:
                fp.write(zf.read(item));fp.flush()
                audio=load_wav(fp.name)
            if len(audio)<(CONTEXT+HMAX+5)*HOP:continue
            records.append({'id':str(path),'group':path.parent.name,'x':spectrogram(audio)})
else:
    for i in range(MAX_RECORDINGS):
        group=['soft','bright','reed','plucked'][i%4]
        tempo=[72,92,118,146,168][(i//4)%5]
        records.append({'id':f'synthetic_{i:02d}_{group}_{tempo}', 'group':group,
                        'tempo':tempo,'x':spectrogram(synth_track(i+900,group,tempo))})
assert len(records)>=12,'At least 12 recordings required for recording-disjoint train/val/test'
print('Tracks',len(records),'frames min',min(len(r['x']) for r in records),'groups',sorted(set(r['group'] for r in records)))

In [ ]:
def recording_split(seed):
    rng=np.random.default_rng(seed)
    bins={}
    for r in records:bins.setdefault(r['group'],[]).append(r)
    train,val,test=[],[],[]
    for group,items in sorted(bins.items()):
        shuffled=items.copy();rng.shuffle(shuffled)
        if len(shuffled)<3:raise ValueError('At least three recordings per group are needed')
        nv=max(1,round(len(shuffled)*.17));nt=max(1,round(len(shuffled)*.17))
        val.extend(shuffled[:nv]);test.extend(shuffled[nv:nv+nt]);train.extend(shuffled[nv+nt:])
    s={key:set(r['id'] for r in xs) for key,xs in [('train',train),('validation',val),('test',test)]}
    assert not (s['train']&s['validation'] or s['train']&s['test'] or s['validation']&s['test'])
    return train,val,test,{key:sorted(value) for key,value in s.items()}

def mask_history(x,rng,keep):
    length=len(x)
    mask=(rng.random(length)<keep)
    mask[0]=True
    out=x.copy()
    last=0;elapsed=np.zeros(length,dtype=np.float32)
    for t in range(length):
        if mask[t]:last=t
        else:out[t]=x[last]
        elapsed[t]=(t-last)*DT
    return out,mask.astype(np.float32),elapsed

def create_windows(items,mean,std,seed,keep_rate=None):
    rng=np.random.default_rng(seed)
    inputs,indicators,ages,targets,track_ids=[],[],[],[],[]
    for r in items:
        v=np.clip((r['x']-mean)/std,-6,6).astype(np.float32)
        for j in range(0,len(v)-CONTEXT-HMAX+1,WINDOW_STRIDE):
            x=v[j:j+CONTEXT]
            k=keep_rate if keep_rate is not None else rng.uniform(*TRAIN_KEEP_RANGE)
            obs,m,age=mask_history(x,rng,k)
            inputs.append(obs);indicators.append(m);ages.append(age)
            targets.append(v[j+CONTEXT:j+CONTEXT+HMAX]);track_ids.append(r['id'])
    def tensor(a):return torch.tensor(np.stack(a),dtype=torch.float32)
    return {'x':tensor(inputs),'mask':tensor(indicators),'age':tensor(ages),
            'y':tensor(targets),'recording_ids':track_ids}

def prepare_seed(seed):
    tr,va,te,split=recording_split(seed)
    training=np.concatenate([r['x'] for r in tr],axis=0)
    mean=training.mean(0);std=training.std(0)+.01
    train=create_windows(tr,mean,std,seed+1)
    val=create_windows(va,mean,std,seed+2,keep_rate=.6)
    tests={rate:create_windows(te,mean,std,seed+int(rate*1000)+100,keep_rate=rate)
           for rate in TEST_KEEP_RATES}
    return train,val,tests,split

def batch_loader(ds,shuffle=False):
    return DataLoader(TensorDataset(ds['x'],ds['mask'],ds['age'],ds['y']),
                      batch_size=BATCH,shuffle=shuffle)

example_train,example_val,example_tests,example_split=prepare_seed(SEEDS[0])
print('Example windows: train',len(example_train['x']),'validation',len(example_val['x']),
      'test',len(example_tests[.65]['x']))
assert not set(example_split['train']) & set(example_split['test'])

## 5. Method implementation — joint PDE–ODE integration and event forcing

All trainable methods use 512 historical frames with identical irregular masks, then subsample 128 positions for their encoders to make Colab experiments practical. The proposed method forecasts the entire 96-frame trajectory with a **single** `odeint` call, where both `(u,z)` persist throughout. A model-derived future event-intensity schedule is interpolated at solver times. The event head is never fed observed future transients. The no-event ablation disables event forcing; the no-diffusion ablation disables spectral diffusion; and the no-dynamic-global ablation blocks evolving latent feedback while retaining the same input encoder and event schedule.

The event schedule is a **continuous forcing term**, not a discontinuous jump operator. This avoids claiming a hybrid-jump implementation that we have not written. The stricter discrete-event formulation remains a research extension.

In [ ]:
class EventCoupledDynamics(nn.Module):
    def __init__(self, mode='proposed', latent=18, max_h=HMAX):
        super().__init__()
        self.mode=mode;self.latent=latent;self.max_h=max_h
        self.encoder=nn.GRU(BANDS+2,latent,batch_first=True)
        self.event_head=nn.Linear(latent,max_h)
        self.event_shape=nn.Parameter(.03*torch.randn(BANDS))
        self.reaction=nn.Sequential(nn.Conv1d(1,12,5,padding=2),nn.Tanh(),
                                    nn.Conv1d(12,1,3,padding=1))
        self.coupling=nn.Linear(latent,BANDS,bias=False)
        self.global_ode=nn.Sequential(nn.Linear(latent+2,32),nn.Tanh(),nn.Linear(32,latent))
        self.diff_raw=nn.Parameter(torch.tensor(-2.))

    def predict(self,x,mask,age,horizon=HMAX,method='rk4'):
        assert horizon<=self.max_h
        features=torch.cat([x[:,::MODEL_STRIDE,:],mask[:,::MODEL_STRIDE,None],
                            age[:,::MODEL_STRIDE,None]],dim=-1)
        _,encoded=self.encoder(features);z0=encoded[-1]
        event_logits=self.event_head(z0)[:,:horizon]
        rate=torch.sigmoid(event_logits)
        if self.mode=='no_event':rate=rate*0
        u0=x[:,-1,:]
        # This closure is an ODE vector field; both state components persist for the whole solve.
        def dynamics(t,state):
            u,z=state
            # Linear interpolation of predicted event intensity (no ground-truth event signal).
            idx=torch.clamp(t/DT-1,min=0,max=horizon-1-1e-5)
            left=idx.long();right=torch.clamp(left+1,max=horizon-1)
            a=(idx-left).to(u.dtype)
            current=rate[:,left]*(1-a)+rate[:,right]*a
            pad=F.pad(u[:,None,:],(1,1),mode='replicate')[:,0]
            lap=pad[:,:-2]-2*u+pad[:,2:]
            diffusion=F.softplus(self.diff_raw)*lap
            if self.mode=='no_diffusion':diffusion=diffusion*0
            global_feedback=self.coupling(z) if self.mode!='no_dynamic_global' else torch.zeros_like(u)
            du=self.reaction(u[:,None,:])[:,0]+global_feedback+diffusion
            du=du+current[:,None]*self.event_shape[None,:]
            dz=(self.global_ode(torch.cat([z,u.mean(-1,keepdim=True),current[:,None]],dim=-1))
                if self.mode!='no_dynamic_global' else torch.zeros_like(z))
            return du,dz
        times=torch.arange(horizon+1,device=x.device,dtype=x.dtype)*DT
        outputs=odeint(dynamics,(u0,z0),times,method=method,
                       options={'step_size':DT*4} if method=='rk4' else None)
        return outputs[0][1:].transpose(0,1), event_logits

class NeuralODEControl(nn.Module):
    def __init__(self,latent=24):
        super().__init__()
        self.encoder=nn.GRU(BANDS+2,latent,batch_first=True)
        self.init=nn.Linear(latent,BANDS)
        self.field=nn.Sequential(nn.Linear(BANDS+latent,64),nn.Tanh(),nn.Linear(64,BANDS))
    def predict(self,x,mask,age,horizon=HMAX,method='rk4'):
        inp=torch.cat([x[:,::MODEL_STRIDE,:],mask[:,::MODEL_STRIDE,None],age[:,::MODEL_STRIDE,None]],dim=-1)
        _,enc=self.encoder(inp);z=enc[-1]
        u0=x[:,-1]+.05*self.init(z)
        def fn(t,u):return self.field(torch.cat([u,z],dim=-1))
        times=torch.arange(horizon+1,device=x.device,dtype=x.dtype)*DT
        pred=odeint(fn,u0,times,method=method,
                    options={'step_size':DT*4} if method=='rk4' else None)
        return pred[1:].transpose(0,1),None

class GRUControl(nn.Module):
    def __init__(self,hidden=48):
        super().__init__();self.encoder=nn.GRU(BANDS+2,hidden,batch_first=True)
        self.head=nn.Linear(hidden,HMAX*BANDS)
    def predict(self,x,mask,age,horizon=HMAX,method='rk4'):
        features=torch.cat([x[:,::MODEL_STRIDE,:],mask[:,::MODEL_STRIDE,None],age[:,::MODEL_STRIDE,None]],-1)
        _,h=self.encoder(features)
        return self.head(h[-1]).reshape(-1,HMAX,BANDS)[:,:horizon],None

class TCNControl(nn.Module):
    def __init__(self,hidden=48):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(BANDS+2,hidden,5,padding=2),nn.ReLU(),
                               nn.Conv1d(hidden,hidden,5,padding=2),nn.ReLU(),nn.AdaptiveAvgPool1d(1))
        self.head=nn.Linear(hidden,HMAX*BANDS)
    def predict(self,x,mask,age,horizon=HMAX,method='rk4'):
        features=torch.cat([x[:,::MODEL_STRIDE,:],mask[:,::MODEL_STRIDE,None],age[:,::MODEL_STRIDE,None]],-1)
        h=self.net(features.transpose(1,2)).squeeze(-1)
        return self.head(h).reshape(-1,HMAX,BANDS)[:,:horizon],None

class TransformerControl(nn.Module):
    def __init__(self,d=48):
        super().__init__()
        self.input=nn.Linear(BANDS+2,d);self.position=nn.Parameter(torch.zeros(1,CONTEXT//MODEL_STRIDE,d))
        layer=nn.TransformerEncoderLayer(d,4,dim_feedforward=96,dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=1)
        self.head=nn.Linear(d,HMAX*BANDS)
    def predict(self,x,mask,age,horizon=HMAX,method='rk4'):
        features=torch.cat([x[:,::MODEL_STRIDE,:],mask[:,::MODEL_STRIDE,None],age[:,::MODEL_STRIDE,None]],-1)
        h=self.encoder(self.input(features)+self.position)
        return self.head(h[:,-1]).reshape(-1,HMAX,BANDS)[:,:horizon],None

MODEL_BUILDERS={'PROPOSED_ECPD':lambda:EventCoupledDynamics('proposed'),
                'no_event':lambda:EventCoupledDynamics('no_event'),
                'no_diffusion':lambda:EventCoupledDynamics('no_diffusion'),
                'no_dynamic_global':lambda:EventCoupledDynamics('no_dynamic_global'),
                'neural_ode':NeuralODEControl,'gru':GRUControl,
                'tcn':TCNControl,'transformer':TransformerControl}
for name,builder in MODEL_BUILDERS.items():
    net=builder()
    print(f'{name:20} parameters={sum(p.numel() for p in net.parameters())}')

In [ ]:
# Pre-flight structural tests: no future target is supplied to predict().
# This executes a short forecast and gradient pass before full training.
smoke_x=example_train['x'][:2].to(device)
smoke_m=example_train['mask'][:2].to(device)
smoke_age=example_train['age'][:2].to(device)
for name in ('PROPOSED_ECPD','no_event','no_dynamic_global','neural_ode','gru'):
    candidate=MODEL_BUILDERS[name]().to(device)
    forecast,events=candidate.predict(smoke_x,smoke_m,smoke_age,horizon=8)
    assert forecast.shape==(2,8,BANDS),f'{name} shape {forecast.shape}'
    assert torch.isfinite(forecast).all(),f'{name} non-finite output'
    (forecast.square().mean()).backward()
    assert any(p.grad is not None for p in candidate.parameters()),name
    print(name,'shape and gradients OK; joint future predictions use only historical inputs')
    del candidate


## 6. Optimization, chronological targets, and independent validation

Every trainable network minimizes the full 96-frame forecast MSE plus a fixed transient-difference term. Only the proposed and no-diffusion/no-event PDE–ODE networks have an explicit auxiliary event head, which is supervised using spectral-flux proxy labels computed from the training targets. The event threshold is estimated **from the training partition only**. This distinction is included as an ablation/capacity consideration. A validation checkpoint, not the test loss, selects the final epoch. The default is a quick smoke test; no conclusions about long-range music structure should be drawn from two epochs on synthetic data.

In [ ]:
def flux(y,last):
    before=torch.cat([last[:,None,:],y[:,:-1]],dim=1)
    return F.relu(y-before).sum(-1)

def train_flux_threshold(train):
    sample=train['y']
    f=flux(sample,train['x'][:,-1,:])
    return float(torch.quantile(f.flatten(),.80))

def run_prediction(model,ds,device=device,batch_size=24):
    model.eval();all_pred=[];all_event=[]
    start=time.perf_counter()
    with torch.no_grad():
        for a in range(0,len(ds['x']),batch_size):
            b=slice(a,a+batch_size)
            pred,events=model.predict(ds['x'][b].to(device),ds['mask'][b].to(device),ds['age'][b].to(device))
            all_pred.append(pred.cpu())
            if events is not None:all_event.append(events.cpu())
    if device.type=='cuda':torch.cuda.synchronize()
    return torch.cat(all_pred),torch.cat(all_event) if all_event else None,(time.perf_counter()-start)/len(ds['x'])

def train_model(name,ds,val,seed,threshold):
    torch.manual_seed(seed)
    model=MODEL_BUILDERS[name]().to(device)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    best=float('inf');best_state=None
    start=time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        for bx,bm,ba,by in batch_loader(ds,shuffle=True):
            bx,bm,ba,by=(v.to(device) for v in (bx,bm,ba,by))
            opt.zero_grad()
            pred,logits=model.predict(bx,bm,ba)
            mse=F.mse_loss(pred,by)
            transient=F.l1_loss(pred[:,1:]-pred[:,:-1],by[:,1:]-by[:,:-1])
            loss=mse+.12*transient
            if logits is not None and name!='no_event':
                labels=(flux(by,bx[:,-1,:])>threshold).float()
                loss=loss+.025*F.binary_cross_entropy_with_logits(logits,labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step()
        vpred,_,_=run_prediction(model,val)
        score=float((vpred-val['y']).square().mean())
        if np.isfinite(score) and score<best:
            best=score;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print('seed',seed,'model',name,'epoch',epoch+1,'validation_MSE',round(score,6))
    if best_state is None:raise RuntimeError('No finite validation checkpoint')
    model.load_state_dict(best_state);model.eval()
    return model,{'training_seconds':time.perf_counter()-start,'validation_mse':best,
                  'parameters':sum(p.numel() for p in model.parameters())}

## 7. Recording-disjoint evaluation: horizons × irregularity × music-specific error

We report mean squared spectral error and high-frequency error at horizons 8, 24, 48 and 96, plus event-proxy precision/recall/F1 with a one-frame tolerance. Because these are **derived spectrogram flux events**, they do not establish human perceptual note-onset accuracy. Each model also reports training time, inference latency and number of parameters. The figure's first panel deliberately restricts the y-axis to trainable/naive baselines rather than allowing linear extrapolation to dwarf differences near 1 MSE.

The main results are unverified until the notebook runs on the chosen data; all experiment outputs are generated by the following cells.

In [ ]:
def onset_proxy(y,last,threshold):
    return (flux(y,last)>threshold).numpy()

def event_f1(pred,truth,last,threshold,tolerance=1):
    predicted=onset_proxy(pred,last,threshold)
    actual=onset_proxy(truth,last,threshold)
    tp=fp=fn=0
    for p,t in zip(predicted,actual):
        pidx=np.flatnonzero(p);tidx=np.flatnonzero(t);used=set()
        for j in pidx:
            valid=[q for q in tidx if q not in used and abs(q-j)<=tolerance]
            if valid:
                chosen=min(valid,key=lambda q:abs(q-j));used.add(chosen);tp+=1
            else:fp+=1
        fn+=len(tidx)-len(used)
    precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)
    return 2*precision*recall/max(precision+recall,1e-12)

def evaluate_metrics(pred,ds,threshold,latency=0):
    truth=ds['y'];last=ds['x'][:,-1]
    result=[]
    for h in HORIZONS:
        p=pred[:,:h];y=truth[:,:h]
        result.append({'horizon_frames':h,'seconds':h*DT,'mse':float((p-y).square().mean()),
         'high_frequency_mse':float((p[:,:,-BANDS//4:]-y[:,:,-BANDS//4:]).square().mean()),
         'onset_proxy_f1':float(event_f1(p,y,last,threshold)),'inference_seconds_per_window':latency})
    return result

ROWS=[];CHECKPOINTS={};SPLITS={}
for seed in SEEDS:
    train,val,tests,split=prepare_seed(seed);SPLITS[seed]=split
    thr=train_flux_threshold(train)
    for name in MODEL_BUILDERS:
        try:
            model,stats=train_model(name,train,val,seed,thr)
            CHECKPOINTS[(seed,name)]=model.cpu().eval()  # retain for HF shared-subset comparison
            for keep,te in tests.items():
                pred,_,lat=run_prediction(CHECKPOINTS[(seed,name)],te,device=torch.device('cpu'))
                for m in evaluate_metrics(pred,te,thr,lat):
                    ROWS.append({'seed':seed,'model':name,'observation_rate':keep,'status':'evaluated',**stats,**m})
        except Exception as exc:
            print(name,'FAILED:',repr(exc));traceback.print_exc(limit=3)
            ROWS.append({'seed':seed,'model':name,'status':'error','error':repr(exc)})
    for keep,te in tests.items():
        last=te['x'][:,-1];prev=te['x'][:,-2]
        for name,pred in [('persistence',last[:,None,:].expand(-1,HMAX,-1)),
            ('linear_extrapolation',torch.stack([last+(i+1)*(last-prev) for i in range(HMAX)],dim=1))]:
            for m in evaluate_metrics(pred,te,thr):
                ROWS.append({'seed':seed,'model':name,'observation_rate':keep,'status':'evaluated',
                    'training_seconds':0.,'parameters':0,**m})

results=pd.DataFrame(ROWS)
results.to_csv(OUT/'recording_level_all_metrics.csv',index=False)
print(results.groupby(['model','observation_rate','horizon_frames']).mse.agg(['mean','std']).round(5).to_string())

## 8. Evaluation figures: focus on the proposed mechanism and difficult regimes

The first panel compares the joint model specifically to its no-event, no-diffusion and no-dynamic-global ablations and strong neural controls on the same 65% observation-rate subset. The second panel isolates the incremental contribution of event forcing via a **difference-in-loss plot**: negative values favor ECPD. The third examines missingness dependence at the longest horizon. Linear extrapolation is reported in the CSV but omitted from the main axis to avoid compressing all useful comparisons.

In [ ]:
valid=results[results.status=='evaluated']
view=valid[valid.observation_rate==.65]
fig,axes=plt.subplots(1,3,figsize=(18,4.7))
for name,part in view.groupby('model'):
    if name=='linear_extrapolation':continue
    curve=part.groupby('horizon_frames').mse.mean()
    kw={'linewidth':3.2,'marker':'o','color':'#0b59c8'} if name=='PROPOSED_ECPD' else {'linewidth':1.3,'alpha':.75}
    axes[0].plot(curve.index,curve.values,label=name,**kw)
axes[0].set(xlabel='Forecast horizon (frames)',ylabel='Held-out spectral MSE ↓',title='Recording-disjoint horizon error')
axes[0].legend(fontsize=8)
for ablation in ('no_event','no_diffusion','no_dynamic_global','neural_ode','gru','tcn','transformer','persistence'):
    p=view[view.model=='PROPOSED_ECPD'].groupby('horizon_frames').mse.mean()
    q=view[view.model==ablation].groupby('horizon_frames').mse.mean()
    if len(p) and len(q):axes[1].plot(p.index,p.values-q.values,marker='.',label=ablation)
axes[1].axhline(0,color='black',linestyle='--',linewidth=1)
axes[1].set(xlabel='Horizon (frames)',ylabel='ECPD MSE − baseline MSE',title='Proposed-method advantage (< 0 favorable)')
axes[1].legend(fontsize=7)
part=valid[valid.horizon_frames==HMAX]
for name in ('PROPOSED_ECPD','no_event','gru','persistence'):
    q=part[part.model==name].groupby('observation_rate').mse.mean().sort_index()
    if len(q):axes[2].plot(q.index,q.values,marker='o',linewidth=3 if name=='PROPOSED_ECPD' else 1.3,label=name)
axes[2].set(xlabel='Observed fraction of history',ylabel='96-frame MSE ↓',title='Robustness to irregular observations')
axes[2].legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'ecpd_proposed_method_evaluation.png',dpi=160)
plt.show()

## 9. Three pretrained Hugging Face forecasters — protocol and limitations

The three **public, pretrained time-series forecasting** checkpoints are:

1. [IBM TinyTimeMixer research R2](https://huggingface.co/ibm-research/ttm-research-r2) (main revision, 512 context / 96 future steps; research-use noncommercial licence).
2. [Amazon Chronos-Bolt-Tiny](https://huggingface.co/amazon/chronos-bolt-tiny) (quantile forecaster; use median 0.5 quantile).
3. [Amazon Chronos-T5-Tiny](https://huggingface.co/amazon/chronos-t5-tiny) (sample forecaster; use sample median).

A strict common HF subset uses **the same recording-disjoint test windows, 512-frame contexts, 96-frame targets, and eight selected spectral bands**. It also evaluates the proposed method and sequence baselines on that exact subset, rather than comparing incomparable bands or sample counts. All models use forward-filled irregular histories, but only the trained ECPD/sequence models receive the missingness mask. This favors them informationally, and the comparison is therefore an **out-of-domain transfer study**, not architecture-matched evidence. TTM's original training frequency is minute-to-hour, radically different from 16-ms audio frames; its frequency token 0 indicates an out-of-vocabulary resolution. We do **not** artificially pad short contexts: all examples truly have 512 historical time indices.

Public downloads should not require `HF_TOKEN`, which bypasses Colab-secret-vault issues. Individual checkpoint failures record the full exception type and traceback and are excluded from performance ranking. Installing TTM dependencies is optional and isolated to its own cell.

In [ ]:
HF_WINDOWS=8 if FAST_DEV else 48
HF_BANDS=np.linspace(0,BANDS-1,8,dtype=int).tolist()
HF_KEEP=.65
HF_TESTS={seed:prepare_seed(seed)[2][HF_KEEP] for seed in SEEDS}
HF_RECORDS=[]
for seed,ds in HF_TESTS.items():
    idx=np.linspace(0,len(ds['x'])-1,min(HF_WINDOWS,len(ds['x'])),dtype=int)
    sub={key:val[idx] if isinstance(val,torch.Tensor) else [val[i] for i in idx]
         for key,val in ds.items()}
    thr=train_flux_threshold(prepare_seed(seed)[0])
    # Common-subset reference scores using identical bands and trajectories.
    for name in list(MODEL_BUILDERS)+['persistence','linear_extrapolation']:
        if name in MODEL_BUILDERS and (seed,name) not in CHECKPOINTS:continue
        if name=='persistence':p=sub['x'][:,-1:, :].expand(-1,HMAX,-1)
        elif name=='linear_extrapolation':
            last=sub['x'][:,-1];previous=sub['x'][:,-2]
            p=torch.stack([last+(j+1)*(last-previous) for j in range(HMAX)],1)
        else:p,_,_=run_prediction(CHECKPOINTS[(seed,name)],sub,device=torch.device('cpu'))
        HF_RECORDS.append({'seed':seed,'model':name,'status':'evaluated',
          'test_mse_common_bands':float((p[:,:,HF_BANDS]-sub['y'][:,:,HF_BANDS]).square().mean()),
          'windows':len(idx),'bands':len(HF_BANDS),'observation_rate':HF_KEEP})
    print('Shared subset seed',seed,'windows',len(idx),'bands',HF_BANDS)

In [ ]:
# Optional TTM dependency. A package-resolution failure will be logged in the
# per-model HF evaluation cell, not abort the Chronos experiments.
import subprocess
install=subprocess.run([sys.executable,'-m','pip','install','-q','tsfm-public'],
                       check=False,capture_output=True,text=True)
print('TTM dependency status:',install.returncode)
if install.returncode:print('TTM install diagnostic:',install.stderr[-1000:])

In [ ]:
# Isolated HF loading; all errors are visible and recorded. Never invent a score.
import inspect
HF_MODELS=['ibm-research/ttm-research-r2','amazon/chronos-bolt-tiny','amazon/chronos-t5-tiny']
for model_id in HF_MODELS:
    started=time.perf_counter()
    try:
        if model_id.startswith('ibm-research/'):
            from tsfm_public import TinyTimeMixerForPrediction
            net=TinyTimeMixerForPrediction.from_pretrained(model_id,revision='main',token=False).to(device).eval()
        else:
            from chronos import BaseChronosPipeline
            pipe=BaseChronosPipeline.from_pretrained(model_id,device_map=str(device),torch_dtype=torch.float32)
        for seed,ds in HF_TESTS.items():
            idx=np.linspace(0,len(ds['x'])-1,min(HF_WINDOWS,len(ds['x'])),dtype=int)
            sample=ds['x'][idx][:,:,HF_BANDS]
            if model_id.startswith('ibm-research/'):
                with torch.inference_mode():
                    # Explicit OOV resolution; 0 is not a calibrated music-specific token.
                    output=net(past_values=sample.to(device),
                      freq_token=torch.zeros(len(sample),dtype=torch.long,device=device))
                    forecast=output.prediction_outputs[:,:HMAX,:].detach().cpu()
            else:
                # Chronos is univariate; each band is an independent time series.
                stacked=sample.transpose(1,2).reshape(-1,CONTEXT)
                with torch.inference_mode():
                    kw={'num_samples':12} if 'chronos-t5' in model_id else {}
                    samples=pipe.predict(stacked,prediction_length=HMAX,**kw)
                # Bolt: quantiles [N,9,H]; T5: samples [N,S,H]; both use median.
                forecast=samples.median(dim=1).values.reshape(len(sample),len(HF_BANDS),HMAX).transpose(1,2)
            expected=ds['y'][idx][:,:,HF_BANDS]
            if forecast.shape!=expected.shape or not torch.isfinite(forecast).all():
                raise ValueError(f'Invalid output shape/finiteness: {forecast.shape} vs {expected.shape}')
            HF_RECORDS.append({'seed':seed,'model':model_id,'status':'evaluated',
                'test_mse_common_bands':float((forecast-expected).square().mean()),
                'windows':len(idx),'bands':len(HF_BANDS),'observation_rate':HF_KEEP,
                'inference_and_load_seconds':time.perf_counter()-started})
        print(model_id,'evaluated successfully')
    except Exception as exc:
        print('Hugging Face MODEL FAILED:',model_id,repr(exc))
        traceback.print_exc(limit=6)
        HF_RECORDS.append({'model':model_id,'status':'load_or_inference_error',
            'error':repr(exc),'traceback':traceback.format_exc(limit=8),
            'inference_and_load_seconds':time.perf_counter()-started})
    finally:
        for var in ('net','pipe'):
            if var in globals():
                del globals()[var]
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
HF_DF=pd.DataFrame(HF_RECORDS)
HF_DF.to_csv(OUT/'hf_three_model_common_subset.csv',index=False)
print(HF_DF[['model','status','test_mse_common_bands']].to_string(index=False))

## 10. Evaluation and visual comparison — exact common subset

The HF comparison includes at least three *attempted* public pretrained checkpoints; a model is counted as evaluated only after producing finite predictions with the expected shape. All available model scores are plotted on the **same eight spectral bands and exact same held-out windows**. ECPD is emphasized with a dark edge and hatching, irrespective of whether it wins. Missing models receive no placeholder bars and are displayed separately as failed.

In [ ]:
hf_good=HF_DF[(HF_DF.status=='evaluated') & HF_DF.test_mse_common_bands.notna()]
fig,ax=plt.subplots(figsize=(12,5))
if len(hf_good):
    means=hf_good.groupby('model').test_mse_common_bands.mean().sort_values()
    bars=ax.bar(np.arange(len(means)),means.values)
    for b,name in zip(bars,means.index):
        if name=='PROPOSED_ECPD':
            b.set_edgecolor('#0b59c8');b.set_linewidth(3);b.set_hatch('///')
    ax.set_xticks(np.arange(len(means)),means.index,rotation=50,ha='right')
    ax.set_ylabel('Common-subset held-out spectral MSE ↓')
    ax.set_title('Proposed method vs sequence and 3 Hugging Face transfer baselines')
else:ax.text(.5,.5,'No valid evaluation results',ha='center',transform=ax.transAxes)
fig.tight_layout();fig.savefig(OUT/'hf_shared_subset_proposed_highlight.png',dpi=160)
plt.show()
missing=HF_DF[HF_DF.status!='evaluated']
if len(missing):print('Non-evaluated checkpoints:',missing[['model','status','error']].to_string(index=False))

## 11. Statistical evaluation and conclusion generator

The notebook distinguishes *scientific outcomes* from implementation status. With a single development seed, the figures are only exploratory. Increase `FAST_DEV=False`, archive recording identifiers, and report confidence intervals before arguing for generalization or efficiency gains. On real recordings, prefer artist- and recording-session-disjoint partitions when metadata allow it. Pretrained time-series models are not assumed competitive on audio by virtue of their success in unrelated domains.

In [ ]:
summary=valid.groupby(['model','horizon_frames','observation_rate']).mse.agg(['mean','std','count']).reset_index()
summary.to_csv(OUT/'aggregated_comparison.csv',index=False)
print('Evidence-based results (96 frames; 65% observations)')
comp=summary[(summary.horizon_frames==96)&(summary.observation_rate==.65)].sort_values('mean')
print(comp[['model','mean','std','count']].to_string(index=False))
ref=comp[comp.model=='PROPOSED_ECPD']
if len(ref):
    base=float(ref['mean'].iloc[0]);competitors=comp[comp.model!='PROPOSED_ECPD']
    if len(competitors):
        best=float(competitors['mean'].min())
        print('Proposed MSE:',base,'best competitor MSE:',best)
        print('Proposed beats best competitor:',bool(base<best))
print('HF successes:',len(HF_DF[(HF_DF.model.isin(HF_MODELS))&(HF_DF.status=='evaluated')]),
      'HF failures:',len(HF_DF[(HF_DF.model.isin(HF_MODELS))&(HF_DF.status!='evaluated')]))
protocol={'seed_list':SEEDS,'context_frames':CONTEXT,'horizons':HORIZONS,'dt_seconds':DT,
  'training_epochs':EPOCHS,'fast_development':FAST_DEV,'observation_rates':TEST_KEEP_RATES,
  'recording_splits':SPLITS,'source_type':'real_WAV' if USE_REAL_WAV_ZIP else 'synthetic',
  'hf_models':HF_MODELS,'hf_common_subset_bands':HF_BANDS,
  'important':'Do not infer cross-recording musical generalization from synthetic-only runs.'}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('Saved experimental protocol and raw data to',OUT)

## 12. Discussion and threats to validity

**Architecture interpretation.** Demonstrated improvements, if any, must be attributed using the no-event, no-diffusion and no-dynamic-global ablations; a low total MSE alone cannot establish that the network learned physically meaningful spectral transport. The history-predicted event schedule offers a concrete explanation for improved transient forecasting only if onset-proxy errors decrease and false-positive events remain controlled.

**Missing data interpretation.** The observation protocol uses missing regular-grid STFT frames and explicit elapsed-time features. This is a faithful test of *irregular availability*, not asynchronous raw audio sampling; no physically irregular waveform acquisition is simulated.

**Numerical validity.** The fixed-step RK4 implementation performs one joint solve per forecast, with output interpolation. Future work should compare adaptive Diffrax/JAX and torchdiffeq solvers, step-size sensitivity, stability, and gradient accuracy. At large horizons, solver interpolation may itself blur transients.

**Out-of-domain pretrained models.** TTM's published checkpoint uses 512 context / 96 steps and was not pretrained at audio temporal frequencies. Chronos is univariate per spectral band and cannot exploit cross-band structure in this protocol. Different use of input masks and historical subsampling prevent claims of strict capacity or information-budget parity. The HF comparison is useful **transfer evidence**, not a decisive architectural leaderboard.

**Data and significance.** Synthetic recordings are not genre-rich real music. Three or fewer test recordings per class can produce high variability. A publishable study requires independent licensed tracks, artist-disjoint splits, >=10 seeds or bootstrap CIs over recordings, properly tuned baselines, and event annotations beyond spectral flux proxies.

## 13. Conclusion and future research

This V3 notebook formulates and implements an event-conditioned, persistent coupled neural PDE–ODE method for forecasting spectra under missing observations. It corrects an identified weakness of the earlier one-step experiment by integrating both field and global state over the complete forecast horizon and applying a long-horizon loss. Its principal contribution at this stage is a **falsifiable controlled evaluation protocol**, not an unverified claim of superior music forecasting.

The next research steps are (i) hybrid jump–flow systems with stable event-sensitive solvers, (ii) JAX/Equinox/Diffrax implementations with explicit solver-performance and gradient comparison, (iii) long-range (>10 s) structural recurrence tasks, (iv) phase- and harmonic-aware nonlocal spectral operators, and (v) a **separate** Weight-Space Programming intervention study of frozen PDE–ODE weights. For that latter direction, evaluation must include null and permuted parameter evidence, deterministic Jacobian repair, and memorization controls following the critical audit of Ahmed (2026). This manuscript-style notebook does not implement LLM-generated weight edits, and the two mechanisms should not be conflated.

## 14. References

1. M. Habiba and B. A. Pearlmutter, *Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE*, 2021. https://arxiv.org/abs/2111.00343
2. R. T. Q. Chen et al., *Neural Ordinary Differential Equations*, NeurIPS 2018. https://arxiv.org/abs/1806.07366
3. P. Kidger et al., *Neural Controlled Differential Equations for Irregular Time Series*, NeurIPS 2020. https://arxiv.org/abs/2005.08926
4. C. Hawthorne et al., *Onsets and Frames: Dual-Objective Piano Transcription*, ISMIR 2018. https://arxiv.org/abs/1710.11153
5. C. Hawthorne et al., *The MAESTRO Dataset*, ICLR 2019. https://arxiv.org/abs/1810.12247
6. V. Ekambaram et al., *Tiny Time Mixers*, 2024. https://arxiv.org/abs/2401.03955
7. A. F. Ansari et al., *Chronos: Learning the Language of Time Series*, 2024. https://arxiv.org/abs/2403.07815
8. N. S. Ahmed, *Can an LLM Actually Reason and Program about Neural Network's Weights or is it just Memorization on Steroids? Exploring Weight-Space Programming*, arXiv:2611.08070v1 (2026). https://arxiv.org/abs/2611.08070
9. J. Hu et al., *Exploring Weight-Space Programming: Capabilities, Generalization, and Limitations*, arXiv:2611.09382 (2026). https://arxiv.org/abs/2611.09382
10. R. Takehi, J. Teoh, A. Zhang and O. Khattab, *Sparks of Weight-Space Programming* (2026), technical report/blog with experiment logs. https://rikiyat.github.io/blog/weight-space-programming/
11. `torchdiffeq` implementation: https://github.com/rtqichen/torchdiffeq
12. Hugging Face TTM-R2 model card: https://huggingface.co/ibm-research/ttm-research-r2
13. Hugging Face Chronos-Bolt-Tiny model card: https://huggingface.co/amazon/chronos-bolt-tiny
14. Hugging Face Chronos-T5-Tiny model card: https://huggingface.co/amazon/chronos-t5-tiny